# Amazon Category Registry

This notebook inspects the versioned category registry built from every product in the Amazon Reviews 2023 Beauty and Personal Care item-metadata file. It replaces the obsolete idea of guessing a generic domain from review keywords.

## Objectives

- reconcile the registry with the complete registered product source;
- distinguish `dataset_category`, `main_category`, and hierarchical `categories`;
- inspect missing metadata, path depth, popular paths, and label normalization;
- define safe rules for selecting a product niche;
- inspect conditioner-related candidates without prematurely locking the MVP niche.

# Реестр категорий Amazon

Этот ноутбук исследует версионный реестр категорий, построенный по всем товарам из Amazon Reviews 2023 Beauty and Personal Care. Он заменяет устаревшую идею угадывать универсальный домен по ключевым словам в отзывах: классификация уже дана Amazon в метаданных товаров, поэтому наша задача — правильно ее сохранить и проверить.

## Цели

- сверить реестр со всем зарегистрированным источником товаров;
- различить `dataset_category`, `main_category` и иерархический путь `categories`;
- изучить пропуски, глубину дерева, популярные пути и нормализацию названий;
- определить безопасные правила выбора товарной ниши;
- посмотреть кандидатов, связанных с кондиционерами, но пока не фиксировать MVP-нишу.

## Inputs and outputs

The reusable builder `src.analytics.category_registry` has already scanned the complete compressed item-metadata source and produced a small registered Parquet plus an exact JSON quality report. This notebook reads those outputs; it does not rescan 711 MB of compressed metadata during every visual review.

No new artifact is written here. The next catalog stage will map every `parent_asin` to these category paths and then measure review coverage.

## Входы и результаты

Переиспользуемый построитель `src.analytics.category_registry` уже просканировал весь сжатый источник метаданных и создал небольшой зарегистрированный Parquet вместе с точным JSON-отчетом. Ноутбук читает готовые результаты и не сканирует заново 711 МБ сжатых данных при каждом визуальном просмотре.

Новые артефакты здесь не создаются. На следующем этапе полный каталог свяжет каждый `parent_asin` с путями категорий, после чего мы измерим покрытие отзывов метаданными.

In [ ]:
# Standard library / Стандартная библиотека
import json
import sys
from pathlib import Path

# Third-party packages / Сторонние библиотеки
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
from IPython.display import display

# Locate the repository root before importing local project modules.
# Находим корень репозитория до импорта локальных модулей проекта.
PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "PLAN.md").is_file() and (candidate / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Local project modules / Локальные модули проекта
from src.analytics.category_registry import (
    CATEGORY_REGISTRY_SCHEMA,
    category_path_key,
    normalize_category_display_label,
    normalize_category_label,
)
from src.common.project import find_project_root
from src.ingestion.dataset_manifest import (
    load_dataset_manifest,
    manifest_is_valid,
    verify_dataset_manifest,
)

In [ ]:
PROJECT_ROOT = find_project_root(PROJECT_ROOT)
MANIFEST_PATH = (
    PROJECT_ROOT
    / "config/datasets/amazon_reviews_2023_beauty_2021_2023_v1.json"
)
CATEGORY_REPORT_PATH = (
    PROJECT_ROOT
    / "reports/data_quality/amazon_reviews_2023_beauty_2021_2023_v1_category_registry.json"
)
TOP_CATEGORY_COUNT = 15

manifest = load_dataset_manifest(MANIFEST_PATH)
registry_registration = manifest.file_by_role("category_registry")
CATEGORY_REGISTRY_PATH = PROJECT_ROOT / registry_registration.path

print(f"Python executable: {sys.executable}")
print(f"DuckDB version: {duckdb.__version__}")
print(f"Dataset category: {manifest.dataset_category}")
print(f"Registry schema: {manifest.schema_versions.category_registry}")

## 1. Three classification levels

The three source fields answer different questions and must not be treated as interchangeable.

| Field | Meaning | Project use |
|---|---|---|
| `dataset_category` | Which Amazon Reviews 2023 download file we selected | Dataset identity and broad processing boundary |
| `main_category` | A broad storefront label stored on an item | Quality signal and optional facet; may be missing or inconsistent |
| `categories` | Ordered Amazon product taxonomy path | Primary source for category/niche selection |

## Три уровня классификации

Три поля источника отвечают на разные вопросы, поэтому их нельзя считать взаимозаменяемыми.

| Поле | Смысл | Использование в проекте |
|---|---|---|
| `dataset_category` | Какой файл Amazon Reviews 2023 мы скачали | Идентичность датасета и широкая граница обработки |
| `main_category` | Широкая storefront-метка в записи товара | Сигнал качества и дополнительный фильтр; может отсутствовать или быть непоследовательным |
| `categories` | Упорядоченный путь товарной классификации Amazon | Основной источник выбора категории и ниши |

In [ ]:
if not CATEGORY_REPORT_PATH.is_file():
    raise FileNotFoundError(CATEGORY_REPORT_PATH)

# Fast size verification avoids rescanning every multi-gigabyte source file.
# Быстрая проверка размеров не сканирует повторно все многогигабайтные источники.
manifest_results = verify_dataset_manifest(
    manifest,
    project_root=PROJECT_ROOT,
    verify_checksums=False,
    verify_record_counts=False,
)
if not manifest_is_valid(manifest_results):
    raise ValueError("One or more registered dataset artifacts are invalid")

category_report = json.loads(CATEGORY_REPORT_PATH.read_text(encoding="utf-8"))
category_registry = pd.read_parquet(CATEGORY_REGISTRY_PATH)
physical_registry_schema = pq.ParquetFile(CATEGORY_REGISTRY_PATH).schema_arrow

assert category_report["input_rows"] == (
    category_report["distinct_parent_asin_count"]
    + category_report["duplicate_parent_asin_rows"]
    + category_report["missing_parent_asin_rows"]
)
assert len(category_registry) == category_report["registry_row_count"]
assert len(category_registry) == registry_registration.record_count
assert physical_registry_schema.equals(
    CATEGORY_REGISTRY_SCHEMA, check_metadata=False
)
assert not category_registry[CATEGORY_REGISTRY_SCHEMA.names].isna().any().any()
assert category_registry["category_path_id"].is_unique
assert category_registry["category_path_key"].is_unique
assert category_registry["product_record_count"].sum() == (
    category_report["input_rows"]
    - category_report["missing_category_path_rows"]
)

display(
    pd.Series(
        {
            "source_product_rows": category_report["input_rows"],
            "distinct_parent_asins": category_report["distinct_parent_asin_count"],
            "missing_parent_asin_rows": category_report["missing_parent_asin_rows"],
            "missing_main_category_rows": category_report["missing_main_category_rows"],
            "missing_category_path_rows": category_report["missing_category_path_rows"],
            "distinct_category_paths": category_report["distinct_category_path_count"],
            "hierarchical_category_nodes": category_report["category_node_count"],
            "physical_schema_and_nullability_passed": True,
            "registry_reconciliation_passed": True,
        },
        name="value",
    ).to_frame()
)

## 2. `main_category` quality

`main_category` is not reliable enough to define the Beauty niche by itself. It is missing for about one tenth of products, and products with Beauty taxonomy paths may carry broad labels from other storefront areas. We preserve this source field, but we do not overwrite the declared dataset category or the hierarchical path with it.

## Качество `main_category`

Одного `main_category` недостаточно для надежного определения Beauty-ниши. Поле отсутствует примерно у десятой части товаров, а товары с Beauty-путями могут иметь широкие метки из других разделов магазина. Мы сохраняем исходное значение, но не заменяем им объявленную категорию датасета или иерархический путь.

In [ ]:
main_category_distribution = pd.DataFrame(
    category_report["main_category_distribution"]
)
main_category_distribution["display_category"] = (
    main_category_distribution["main_category"].fillna("<missing>")
)
main_category_distribution["product_share_pct"] = (
    main_category_distribution["product_record_count"]
    / category_report["input_rows"]
    * 100
)
display(main_category_distribution.head(TOP_CATEGORY_COUNT))

sns.set_theme(style="whitegrid")
plot_frame = main_category_distribution.head(10).sort_values(
    "product_record_count"
)
figure, axis = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=plot_frame,
    x="product_record_count",
    y="display_category",
    color="#4C78A8",
    ax=axis,
)
axis.set_title("Top Source main_category Labels / Основные метки main_category")
axis.set_xlabel("Product records / Товарные записи")
axis.set_ylabel("main_category")
plt.tight_layout()
plt.show()

## 3. Hierarchical category paths

A path is an ordered list from a broad category to a specific leaf. The registry stores one row per distinct path and aggregates product counts across inconsistent `main_category` values. `category_node_count` counts distinct path prefixes, so two identically named leaves under different parents remain different nodes.

## Иерархические пути категорий

Путь — это упорядоченный список от широкой категории к конкретному листу дерева. В реестре одна строка соответствует одному уникальному пути, а количество товаров суммируется независимо от непоследовательного `main_category`. `category_node_count` считает уникальные префиксы пути, поэтому одинаково названные листья под разными родителями остаются разными узлами.

In [ ]:
depth_distribution = pd.DataFrame(
    category_report["category_depth_distribution"]
)
top_paths = category_registry.nlargest(
    TOP_CATEGORY_COUNT, "product_record_count"
)[
    [
        "category_path_id",
        "category_path_text",
        "category_depth",
        "product_record_count",
        "main_category_count",
    ]
]
display(depth_distribution)
display(top_paths)

figure, axis = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=depth_distribution,
    x="category_depth",
    y="product_record_count",
    color="#72B7B2",
    ax=axis,
)
axis.set_title("Exact Product Count by Path Depth / Товары по глубине пути")
axis.set_xlabel("Category depth / Глубина категории")
axis.set_ylabel("Product records / Товарные записи")
plt.tight_layout()
plt.show()

## 4. Display labels versus matching keys

Display labels preserve source wording and casing after Unicode NFKC normalization, whitespace collapsing, and edge trimming. Empty labels are removed. Separate matching keys additionally use case folding. The registry and product catalog share these functions, so the same source path cannot acquire different technical identities. Normalization does not translate labels, merge synonyms, or invent a new taxonomy.

## Отображаемые названия и ключи поиска

Отображаемые названия сохраняют исходные слова и регистр после Unicode NFKC, схлопывания пробелов и обрезки краев; пустые элементы удаляются. Ключи поиска дополнительно используют case folding. Реестр и каталог вызывают одни и те же функции, поэтому один исходный путь не получает разные технические идентификаторы. Нормализация не переводит названия, не объединяет синонимы и не придумывает новую классификацию.

In [ ]:
# The third label uses full-width Unicode letters. They look like Latin
# letters but have different code points; NFKC converts them to ASCII form.
# В третьем примере используются полноширинные Unicode-буквы. Они похожи на
# латинские, но имеют другие кодовые точки; NFKC приводит их к форме ASCII.
normalization_examples = pd.DataFrame(
    {
        "example": [
            "already normalized",
            "repeated whitespace",
            "full-width Unicode letters",
        ],
        "source_label": [
            "Hair Care",
            "  Hair   Care  ",
            "Ｔｏｏｌｓ & Accessories",
        ]
    }
)
normalization_examples["matching_key"] = normalization_examples["source_label"].map(
    normalize_category_label
)
normalization_examples["display_label"] = normalization_examples["source_label"].map(
    normalize_category_display_label
)
normalized_paths_match = category_registry.apply(
    lambda row: (
        list(row["category_path"])
        == [
            normalize_category_display_label(label)
            for label in row["category_path"]
        ]
        and row["category_path_text"] == " > ".join(row["category_path"])
        and row["category_path_key"]
        == category_path_key(list(row["category_path"]))
    ),
    axis=1,
).all()
assert category_registry["category_path_key"].is_unique
assert normalized_paths_match
display(normalization_examples)

## 5. Safe niche-selection rules

A niche is not a free-text keyword and not an LLM guess. For reproducible analytics it is a reviewed, versioned set of `category_path_id` values.

### English

1. Search the registry to discover candidate paths.
2. Inspect their full hierarchy and product examples.
3. Explicitly include or exclude related leaves; substring search is discovery only.
4. Save approved stable path IDs in versioned niche configuration.
5. Use the full catalog to map paths to `parent_asin`, then join all eligible reviews.
6. Record dataset version, filters, product count, review count, and coverage.

## Правила безопасного выбора ниши

Ниша — это не произвольное ключевое слово и не догадка LLM. Для воспроизводимой аналитики ниша задается проверенным версионным набором `category_path_id`.

### Русский

1. Найти пути-кандидаты в реестре.
2. Проверить их полную иерархию и примеры товаров.
3. Явно включить или исключить связанные листья; поиск по подстроке нужен только для обнаружения кандидатов.
4. Сохранить утвержденные стабильные ID путей в версионной конфигурации ниши.
5. Через полный каталог получить `parent_asin`, затем присоединить все подходящие отзывы.
6. Зафиксировать версию данных, фильтры, количество товаров, отзывов и покрытие.

In [ ]:
# Search leaf labels first; matching the whole path would also return shampoos
# merely because their parent node is named 'Shampoo & Conditioner'.
# Сначала ищем по названию листа: поиск по всему пути вернул бы также шампуни
# только из-за родительского узла 'Shampoo & Conditioner'.
conditioner_candidates = category_registry[
    category_registry["leaf_category_key"].str.contains(
        "conditioner", regex=False
    )
].sort_values("product_record_count", ascending=False)
display(
    conditioner_candidates[
        [
            "category_path_id",
            "category_path_text",
            "product_record_count",
            "product_share",
        ]
    ]
)

## 6. Interpretation and next decision

The displayed full-population reconciliation shows that every registered parent product has a non-empty normalized category path in this snapshot. The exact path, prefix-node, depth, and missing-`main_category` counts are reported above rather than duplicated as manually maintained constants. The broad `main_category` still varies widely, confirming that the ordered `categories` path should drive niche selection.

Conditioners remain a plausible MVP candidate, but the displayed paths show that 'conditioner' covers several materially different niches: standard conditioners, deep conditioners, 2-in-1 products, beard products, color conditioners, and more. The full catalog in notebook 05 now provides product examples and review coverage; after reviewing them we can approve the appropriate path set.

## Интерпретация и следующее решение

Показанная сверка по полной совокупности подтверждает, что в этом срезе у каждого зарегистрированного родительского товара есть непустой нормализованный путь категории. Точные количества путей, узлов-префиксов, глубин и пропусков `main_category` приведены выше и не дублируются вручную в тексте. Широкий `main_category` по-прежнему сильно варьируется, поэтому выбор ниши должен опираться прежде всего на упорядоченный путь `categories`.

Кондиционеры остаются разумным кандидатом для MVP, но таблица показывает, что слово conditioner охватывает разные ниши: обычные и глубокие кондиционеры, средства 2-в-1, продукты для бороды, кондиционеры для окрашенных волос и другие. Полный каталог в ноутбуке 05 уже показывает примеры товаров и покрытие отзывами; после их просмотра можно решить, какой набор путей утвердить.